In [ ]:
from IPython.display import display, HTML
try:
    from notebook.services.config import ConfigManager
    cm = ConfigManager()
    cm.update('livereveal', {
        'theme': 'simple',
        'transition': 'fade',
        'start_slideshow_at': 'selected',
    })
except Exception:
    pass
display(HTML('<style>.rise-enabled .cell { font-size: 90%; }</style>'))

# Week 09: Wednesday, AST 5011: Astrophysical Systems

## The Formation of Dark Matter Halos

### Michael Coughlin

**Reference:** Cimatti, Fraternali & Nipoti, Ch. 4

In [ ]:
import numpy as np
import scipy.special
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, MultipleLocator
import matplotlib.gridspec as gridspec

from colossus.cosmology import cosmology
from colossus.lss import peaks
from colossus.lss import mass_function
from colossus.halo import mass_so
from colossus.halo import concentration
from colossus.halo import profile_nfw, profile_einasto, profile_hernquist
from colossus.halo import profile_composite
from colossus.utils import constants

import routines

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

cosmo = routines.cosmo

---

## Non-linear Growth and Halo Collapse

In linear perturbation theory, density perturbations grow as $\delta \propto D(a)$, where $D(a)$ is the linear growth factor. But linear theory breaks down when $\delta \sim 1$. To understand what happens next, we turn to the **spherical (top-hat) collapse model** -- the simplest fully non-linear calculation in structure formation.

### The Spherical Collapse Model

Consider a uniform spherical overdensity embedded in the expanding background. Because it is overdense, it behaves like a small closed universe:

1. **Expansion phase:** The overdense sphere initially expands with the Hubble flow, but more slowly than the background.
2. **Turnaround:** At some point the sphere reaches a maximum radius and stops expanding. At turnaround, the *actual* overdensity is $\delta \approx 4.6$ (in an Einstein-de Sitter universe).
3. **Collapse:** The sphere then contracts under its own gravity. In the idealized calculation it collapses to a point, but in practice the material virializes at roughly half the turnaround radius.

### Why $\delta_c = 1.686$?

The critical value $\delta_c = 1.686$ is the **linearly extrapolated** overdensity at the moment of collapse. This means: if we take the initial perturbation and evolve it using *linear* theory all the way to the collapse time, we get $\delta_{\rm linear} = 1.686$, even though the true non-linear overdensity has become infinite (or, physically, the object has virialized with $\delta_{\rm true} \sim 178$).

Why use this peculiar definition? Because the statistical tools we have (the variance $\sigma(M)$ of the density field) are computed in **linear theory**. By expressing the collapse threshold in linear units, we can directly compare $\delta_c$ to $\sigma(M)$ and ask: what fraction of the universe has collapsed?

At any redshift $z$, there is a characteristic mass scale where the *average* variance $\sigma(M)$ of the density field equals $\delta_c$. This defines the **non-linear mass** $M_{\rm nl}$: the typical mass of halos forming at that epoch. More precisely, we define the **peak height**:

$$\nu \equiv \frac{\delta_c}{\sigma(M, z)}$$

A region with $\nu = 1$ is a typical ($1\sigma$) fluctuation that is just collapsing. Regions with $\nu \gg 1$ are rare, high-sigma peaks.

Structure formation is **bottom-up** (hierarchical): since $\sigma(M)$ is larger on smaller scales (the power spectrum has more power on small scales in CDM cosmologies), low-mass halos form first. The massive clusters we see today assembled relatively recently from the mergers of smaller halos.

In [ ]:
# Non-linear mass today
M_nl_today = peaks.nonLinearMass(0.0) / cosmo.h
print('The non-linear mass today is %.1e Msun.' % M_nl_today)

# Evolution of mass scales at different peak heights
peak_heights = [4, 2, 1, 0.5]
cmap = plt.get_cmap('viridis')

a = np.linspace(0.06, 1.5, 100)
z = 1.0 / a - 1.0

plt.figure(figsize=(5, 4))
ax = plt.gca()
plt.yscale('log')
plt.xlabel(r'$a$')
plt.ylabel(r'$M\ (M_\odot)$')
plt.xlim(a[0], a[-1])
plt.ylim(1e6, 1e16)
ax.xaxis.set_major_locator(MultipleLocator(0.2))
ax.xaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.set_major_locator(LogLocator(numticks=100))
ax.yaxis.set_minor_locator(LogLocator(numticks=100, subs=np.arange(2, 10)))
plt.grid(lw=0.5, ls=':')

plt.axhline(M_nl_today, ls='--', color='gray', lw=0.8)
plt.axvline(1.0, ls='--', color='gray', lw=0.8)

for i, nu in enumerate(peak_heights):
    M = peaks.massFromPeakHeight(nu, z) / cosmo.h
    c = cmap(float(i) / (len(peak_heights) - 1.0))
    plt.plot(a, M, color=c, label=r'$\nu = %s$' % str(nu))

plt.legend(frameon=True)
plt.title('Mass scales vs. scale factor')
plt.tight_layout()
plt.show()


The curves show the mass scale at which $\nu = \delta_c / \sigma(M, z) = $ const. The **non-linear mass** ($\nu = 1$) is the typical mass collapsing at each epoch. This single plot encapsulates the entire history of hierarchical structure formation.

Key observations:
- At early times ($a \ll 1$), only very low-mass halos have formed -- the first collapsed objects are tiny.
- By $a = 1$ (today), halos up to $\sim 10^{13}\,M_\odot$ are collapsing.
- Rare, massive clusters correspond to $\nu \gg 1$ (the exponential tail of the mass function). A $10^{15}\,M_\odot$ cluster at $z = 0$ is a $\sim 3\sigma$ peak -- rare but not extraordinary. The same mass at $z = 2$ would be a $\sim 5\sigma$ fluctuation -- essentially impossible.
- This is why finding massive galaxy clusters at high redshift is such a powerful test of cosmology.

---

## The Press-Schechter Mass Function

How many dark matter halos of a given mass exist at a given redshift? **Press & Schechter (1974)** answered this with a remarkably simple argument that connects Gaussian random fields to halo abundances. Let us walk through the logic step by step.

### The Key Idea

The density field $\delta(\mathbf{x})$ at early times is a **Gaussian random field** -- its values at any point are drawn from a Gaussian distribution. When we smooth this field on a scale $R$ (corresponding to mass $M \propto R^3$), the smoothed field is also Gaussian with variance $\sigma^2(M)$.

**The Press-Schechter ansatz:** A region of the universe ends up in a halo of mass $> M$ if the smoothed density at that point exceeds the collapse threshold:

$$\delta_R(\mathbf{x}) > \delta_c$$

Since $\delta_R$ is Gaussian with variance $\sigma^2(M)$, the fraction of the universe in halos more massive than $M$ is simply:

$$F(>M) = \frac{1}{2}\,\mathrm{erfc}\left(\frac{\nu}{\sqrt{2}}\right) \quad \text{where } \nu = \frac{\delta_c}{\sigma(M)}$$

There is a subtlety: this integral only accounts for half the mass in the universe (the half with $\delta > 0$). Press & Schechter multiplied by a **"fudge factor" of 2**, arguing that underdense regions will eventually be accreted by neighboring overdensities. This factor was later rigorously justified by the **excursion set** (Bond et al. 1991) formalism.

### The Multiplicity and Mass Functions

Differentiating $F(>M)$ with respect to mass gives the **multiplicity function** (fraction of mass in halos per unit $\ln\nu$):

$$f_{\rm PS}(\nu) = \sqrt{\frac{2}{\pi}}\, \nu\, \exp\left(-\frac{\nu^2}{2}\right)$$

The **mass function** (number density of halos per unit $\ln M$) is then:

$$\frac{dn}{d\ln M} = \frac{\bar{\rho}}{M}\, f(\nu)\, \left|\frac{d\ln\sigma}{d\ln M}\right|$$

This expression has a beautifully intuitive structure: $\bar{\rho}/M$ gives the number density if *all* mass were in halos of mass $M$, $f(\nu)$ gives the fraction actually in halos at this peak height, and $|d\ln\sigma/d\ln M|$ converts from $\nu$-space to $M$-space.

In [ ]:
# Press-Schechter multiplicity function
nu = np.linspace(0.22, 5.0, 50)
f_PS = np.sqrt(2.0 / np.pi) * nu * np.exp(-nu**2 / 2.0)

plt.figure(figsize=(5, 4))
plt.loglog()
plt.xlim(nu[0], nu[-1])
plt.xlabel(r'$\nu$')
plt.ylabel(r'$f(\nu)$')
plt.plot(nu, f_PS, lw=2)
plt.title('Press-Schechter multiplicity function')
plt.tight_layout()
plt.show()

In [ ]:
# Mass function at multiple redshifts (using Colossus)
zs = [0, 1, 2, 4, 7]
cmap = plt.get_cmap('viridis')
M = 10**np.linspace(7.0, 15.0, 100)

plt.figure(figsize=(5, 4))
plt.loglog()
plt.xlim(M[0], M[-1])
plt.ylim(1e-6, 8e2)
plt.xlabel(r'$M\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$dn / d\ln(M)\ (h^3\, {\rm Mpc}^{-3})$')
plt.grid(lw=0.5, ls=':')

for i, zi in enumerate(zs):
    c = cmap(float(i) / (len(zs) - 1.0))
    dn = mass_function.massFunction(M, zi, q_in='M', q_out='dndlnM', model='press74')
    plt.plot(M, dn, c=c, label=r'$z = %s$' % str(zi))

plt.legend()
plt.title('Press-Schechter mass function')
plt.tight_layout()
plt.show()

---

### Exercise 1: Computing the Halo Mass Function

In this exercise, you will compute the Press-Schechter mass function **from scratch** using the building blocks from Colossus, and compare to the more accurate Tinker et al. (2008) fitting function.

**Steps:**
1. Convert masses $M$ to Lagrangian radii $R_L = (3M / 4\pi\bar{\rho})^{1/3}$. This is the comoving radius that encloses mass $M$ at the mean density -- it sets the smoothing scale of the density field.
2. Compute $\sigma(R_L)$ at $z = 0$ using `cosmo.sigma(R_L)`. This is the rms density fluctuation on scale $R_L$.
3. Compute the peak height $\nu = \delta_c / \sigma$. Halos with $\nu > 1$ are forming from above-average fluctuations.
4. Compute the PS multiplicity function $f_{\rm PS}(\nu) = \sqrt{2/\pi}\,\nu\,\exp(-\nu^2/2)$.
5. Compute $|d\ln\sigma / d\ln M|$ using `cosmo.sigma()` with `derivative=True`. (Note: since $R \propto M^{1/3}$, you will need to convert the derivative with respect to $R$ to one with respect to $M$.)
6. Assemble the mass function $dn/d\ln M = (\bar{\rho}/M)\, f(\nu)\, |d\ln\sigma/d\ln M|$.
7. Compare to `mass_function.massFunction()` with `model='tinker08'`.

**Expected results and interpretation:**
- Your PS result should match the Tinker08 curve within a factor of ~2 over most of the mass range.
- At the **low-mass end** ($M \lesssim 10^{11}\,M_\odot$), PS overpredicts -- there are fewer small halos than the simple theory predicts.
- At the **high-mass end** ($M \gtrsim 10^{14}\,M_\odot$), PS underpredicts -- massive clusters are more abundant than PS suggests, because ellipsoidal collapse is easier than spherical collapse for rare peaks.
- **Question to think about:** Why does the mass function steepen so dramatically above $M_{\rm nl}$? What does this tell you about the statistics of Gaussian random fields?

In [ ]:
# Exercise 1: Compute the Press-Schechter mass function from scratch

delta_c = 1.686
M = 10**np.linspace(7.0, 16.0, 100)  # h^-1 Msun

# Step 1: Mean matter density and Lagrangian radii
rho_m_0 = cosmo.rho_m(0.0) * 1e9  # h^2 Msun / Mpc^3
R_L = ...  # FILL IN: (3.0 * M / (4.0 * np.pi * rho_m_0))**(1.0 / 3.0)

# Step 2: Compute sigma(R_L) at z=0
sigma_M = ...  # FILL IN: cosmo.sigma(R_L, z=0.0)

# Step 3: Peak height
nu = ...  # FILL IN: delta_c / sigma_M

# Step 4: PS multiplicity function
f_PS = ...  # FILL IN: np.sqrt(2.0 / np.pi) * nu * np.exp(-nu**2 / 2.0)

# Step 5: |d ln sigma / d ln M|
# cosmo.sigma with derivative=True returns d ln sigma / d ln R; divide by 3 since M ~ R^3
dlnsig_dlnM = ...  # FILL IN: np.abs(cosmo.sigma(R_L, z=0.0, derivative=True) / 3.0)

# Step 6: Mass function
dn_dlnM_PS = ...  # FILL IN: rho_m_0 / M * f_PS * dlnsig_dlnM

# Step 7: Compare to Tinker+2008
dn_dlnM_T08 = mass_function.massFunction(M, 0.0, q_in='M', q_out='dndlnM', model='tinker08', mdef='200m')

# Plot
plt.figure(figsize=(5, 5))
plt.loglog()
plt.xlim(M[0], M[-1])
plt.ylim(1e-8, 1e1)
plt.xlabel(r'$M\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$dn / d\ln(M)\ (h^3\, {\rm Mpc}^{-3})$')
plt.grid(lw=0.5, ls=':')

plt.plot(M, dn_dlnM_PS, '-', lw=2, label='Press-Schechter (yours)')
plt.plot(M, dn_dlnM_T08, '--', lw=2, label='Tinker+2008')

plt.legend()
plt.title('Halo mass function at z = 0')
plt.tight_layout()
plt.show()

# Print the ratio at the high-mass end
print('PS / Tinker08 at M = 1e14 h^-1 Msun:')
idx = np.argmin(np.abs(M - 1e14))
print('  Ratio = %.2f' % (dn_dlnM_PS[idx] / dn_dlnM_T08[idx]))

---

## Halo Density Profiles

One of the most remarkable results from N-body simulations is that dark matter halos have a **universal density profile** -- the same functional form describes halos from dwarf galaxy scales ($10^8\,M_\odot$) to massive cluster scales ($10^{15}\,M_\odot$). The most commonly used models are:

**NFW profile** (Navarro, Frenk & White 1997):
$$\rho(r) = \frac{\rho_s}{(r/r_s)(1 + r/r_s)^2}$$

**Einasto profile:**
$$\rho(r) = \rho_s \exp\left[-\frac{2}{\alpha}\left(\left(\frac{r}{r_s}\right)^\alpha - 1\right)\right]$$

**Hernquist profile** (often used for bulges/ellipticals):
$$\rho(r) = \frac{\rho_s}{(r/r_s)(1 + r/r_s)^3}$$

All three are characterized by a scale radius $r_s$ and scale density $\rho_s$. The **concentration** $c = R_{\rm vir} / r_s$ describes how centrally concentrated the halo is.

### What the Slopes Mean Physically

The NFW profile has three distinct regimes that reveal the physics of halo assembly:

- **Inner region** ($r \ll r_s$): $\rho \propto r^{-1}$ -- a shallow "cusp." This material was deposited early, when the halo's progenitors were small and dense. The gentle slope reflects violent relaxation during early mergers.
- **Intermediate region** ($r \sim r_s$): The profile transitions smoothly through the scale radius. This marks the boundary between material that has completed many orbits and recently accreted material.
- **Outer region** ($r \gg r_s$): $\rho \propto r^{-3}$ -- a steep fall-off. This material was accreted most recently and has not had time to mix inward. The $r^{-3}$ slope matches the infall rate onto the halo.

The Einasto profile improves on NFW by allowing a continuously varying logarithmic slope (controlled by $\alpha$), which better matches the gentle curvature seen in high-resolution simulations. The Hernquist profile falls off more steeply ($r^{-4}$) and has a finite total mass, making it convenient for analytic calculations of stellar systems.

In [ ]:
# Compare density profiles using Colossus
M_vir = 1e14       # h^-1 Msun
c_vir = 5.0
z_plot = 0.0
mdef = '200m'

R_vir = mass_so.M_to_R(M_vir, z_plot, mdef)
rR = 10**np.arange(-2.5, 1.7, 0.02)
r = rR * R_vir
rho_m = cosmo.rho_m(z_plot)

# Create profiles
prf_nfw = profile_composite.compositeProfile(inner_name='nfw', outer_names=[],
                                              M=M_vir, c=c_vir, z=z_plot, mdef=mdef)
prf_ein = profile_composite.compositeProfile(inner_name='einasto', outer_names=[],
                                              M=M_vir, c=c_vir, z=z_plot, mdef=mdef)
prf_her = profile_composite.compositeProfile(inner_name='hernquist', outer_names=[],
                                              M=M_vir, c=c_vir, z=z_plot, mdef=mdef)

fig = plt.figure(figsize=(5, 6))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.8, 0.5])
plt.subplots_adjust(left=0.18, right=0.96, top=0.95, bottom=0.12, hspace=0.06)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Density panel
plt.sca(ax1)
plt.loglog()
plt.ylabel(r'$\rho / \rho_m$', labelpad=8)
ax1.set_xticklabels([])
plt.xlim(rR[0], rR[-1])
plt.ylim(1e-1, 5e6)

for prf, ls, label in [(prf_ein, '-', 'Einasto'), (prf_nfw, '-.', 'NFW'),
                         (prf_her, '--', 'Hernquist')]:
    rho = prf.density(r)
    plt.plot(rR, rho / rho_m, ls=ls, lw=1.4, label=label)

plt.legend(loc=3)
plt.text(0.95, 0.88, r'$M_{200m} = 10^{14}\, h^{-1} M_\odot,\ c = 5$',
         transform=plt.gca().transAxes, fontsize=10, ha='right')

# Slope panel
plt.sca(ax2)
plt.xscale('log')
plt.xlim(rR[0], rR[-1])
plt.ylim(-4.9, 0.3)
plt.xlabel(r'$r / R_{\rm vir}$')
plt.ylabel(r'$d \log(\rho) / d \log(r)$', labelpad=12)
ax2.yaxis.set_major_locator(MultipleLocator(1.0))
ax2.yaxis.set_minor_locator(MultipleLocator(0.5))

for prf, ls, label in [(prf_ein, '-', 'Einasto'), (prf_nfw, '-.', 'NFW'),
                         (prf_her, '--', 'Hernquist')]:
    slope = prf.densityDerivativeLog(r)
    plt.plot(rR, slope, ls=ls, lw=1.4, label=label)

plt.show()

### Realistic Halo Outskirts

The profiles above describe the **orbiting** material within the halo -- particles that have passed through the halo at least once and are on bound orbits. In reality, halos are surrounded by matter falling in for the first time. The [Diemer (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.519.3292D) model captures this by adding an **infalling** power-law envelope beyond $R_{\rm vir}$.

At the transition between orbiting and infalling material, the density profile steepens sharply. This feature is called the **splashback radius** -- the apocentric radius of recently accreted material. It represents a physically meaningful boundary of the halo, in contrast to the somewhat arbitrary spherical overdensity definitions (like $R_{200m}$). We also include the Diemer23 inner profile, which provides a better fit to simulated halos than the classic models.

In [ ]:
# Halo profiles with outer infalling envelope
M_out = 1e14
c_out = 5.0
z_out = 0.0

R_out = mass_so.M_to_R(M_out, z_out, '200m')
rR_out = 10**np.arange(-2.5, 1.7, 0.02)
r_out = rR_out * R_out
rho_m_out = cosmo.rho_m(z_out)

outer_names = ['infalling', 'mean']
prf_names_out = [
    ('einasto', '-', 'Einasto'),
    ('nfw', '-.', 'NFW'),
    ('hernquist', '--', 'Hernquist'),
    ('diemer23', ':', 'Diemer 2023'),
]

fig = plt.figure(figsize=(5, 6))
gs = gridspec.GridSpec(2, 1, height_ratios=[0.8, 0.5])
plt.subplots_adjust(left=0.18, right=0.96, top=0.95, bottom=0.12, hspace=0.06)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Density panel
plt.sca(ax1)
plt.loglog()
plt.ylabel(r'$\rho / \rho_m$', labelpad=8)
ax1.set_xticklabels([])
plt.xlim(rR_out[0], rR_out[-1])
plt.ylim(1e-1, 5e6)

for name, ls, label in prf_names_out:
    prf = profile_composite.compositeProfile(inner_name=name, outer_names=outer_names,
            M=M_out, c=c_out, z=z_out, mdef='200m', pl_delta_1=10.0, pl_s=1.5)
    rho = prf.density(r_out)
    plt.plot(rR_out, rho / rho_m_out, ls=ls, lw=1.4, label=label)

plt.legend(loc=3)
plt.text(0.95, 0.88, r'With outer (infalling) profile',
         transform=plt.gca().transAxes, fontsize=10, ha='right')

# Slope panel
plt.sca(ax2)
plt.xscale('log')
plt.xlim(rR_out[0], rR_out[-1])
plt.ylim(-4.9, 0.3)
plt.xlabel(r'$r / R_{\rm vir}$')
plt.ylabel(r'$d \log(\rho) / d \log(r)$', labelpad=12)
ax2.yaxis.set_major_locator(MultipleLocator(1.0))
ax2.yaxis.set_minor_locator(MultipleLocator(0.5))

for name, ls, label in prf_names_out:
    prf = profile_composite.compositeProfile(inner_name=name, outer_names=outer_names,
            M=M_out, c=c_out, z=z_out, mdef='200m', pl_delta_1=10.0, pl_s=1.5)
    slope = prf.densityDerivativeLog(r_out)
    plt.plot(rR_out, slope, ls=ls, lw=1.4, label=label)

plt.show()

### The Concentration-Mass Relation

The concentration $c = R_{\rm vir}/r_s$ encodes the **formation history** of a halo. The physical picture is:

1. When a halo first forms, its scale radius $r_s$ is set by the density of the universe at that time. Denser universe $\Rightarrow$ smaller $r_s$ $\Rightarrow$ higher concentration.
2. After formation, the halo grows by accreting material onto its outskirts. This increases $R_{\rm vir}$ but leaves the inner profile (and $r_s$) largely unchanged.
3. Therefore $c = R_{\rm vir}/r_s$ reflects the **ratio of the current size to the size at formation**.

This leads to the observed correlation with mass:
- **Low-mass halos** (e.g., $M \sim 10^{11}\,M_\odot$) formed earlier, when the universe was denser $\Rightarrow$ higher concentration ($c \approx 8$-$12$ at $z=0$).
- **Massive halos** (e.g., $M \sim 10^{15}\,M_\odot$) formed recently $\Rightarrow$ lower concentration ($c \approx 4$-$6$ at $z=0$).
- At **higher redshift**, all halos are younger and less evolved, so concentrations are lower across the board.

The median concentration at $z = 0$ drops from $c \approx 10$ for Milky Way-mass halos to $c \approx 5$ for cluster-mass halos. The scatter around the median is $\sim 0.15$ dex, reflecting the diversity of assembly histories.

In [ ]:
# Concentration-mass relation at multiple redshifts
M_cm = 10**np.linspace(10, 15, 50)
zs_cm = [0.0, 0.5, 1.0, 2.0]
cmap = plt.get_cmap('viridis')

plt.figure(figsize=(5, 4))
plt.xscale('log')
plt.xlabel(r'$M_{\rm vir}\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$c_{\rm vir}$')
plt.xlim(M_cm[0], M_cm[-1])
plt.ylim(2, 20)

for i, zi in enumerate(zs_cm):
    c_arr = concentration.concentration(M_cm, 'vir', zi)
    color = cmap(float(i) / (len(zs_cm) - 1.0))
    plt.plot(M_cm, c_arr, color=color, lw=1.5, label=r'$z = %.1f$' % zi)

plt.legend()
plt.title('Concentration-mass relation (Diemer & Joyce 2019)')
plt.tight_layout()
plt.show()

### Enclosed Mass and Circular Velocity

Different density profile models predict different enclosed mass distributions $M(<r)$ and therefore different circular velocity curves:

$$V_c(r) = \sqrt{\frac{GM(<r)}{r}}$$

This is the critical link between theory and observation: the density profile is not directly observable, but the circular velocity curve *is* -- from rotation curves of gas disks, stellar kinematics, or the motions of satellite galaxies.

For the NFW profile, $V_c(r)$ peaks at approximately $r \approx 2.16\, r_s$ and then declines slowly. The location and height of this peak directly constrain the halo's mass and concentration. Profiles with higher concentration peak at smaller radii (relative to $R_{\rm vir}$) and have a steeper decline at large radii.

In [ ]:
# Enclosed mass and circular velocity for different profile models
M_prof = 1e14       # h^-1 Msun
c_prof = 5.0
z_prof = 0.0

R_prof = mass_so.M_to_R(M_prof, z_prof, '200m')
rR_prof = 10**np.arange(-2.5, 1.2, 0.02)
r_prof = rR_prof * R_prof

# Create profiles
profiles = [
    (profile_composite.compositeProfile(inner_name='nfw', outer_names=[],
        M=M_prof, c=c_prof, z=z_prof, mdef='200m'), '-.', 'NFW'),
    (profile_composite.compositeProfile(inner_name='einasto', outer_names=[],
        M=M_prof, c=c_prof, z=z_prof, mdef='200m'), '-', 'Einasto'),
    (profile_composite.compositeProfile(inner_name='hernquist', outer_names=[],
        M=M_prof, c=c_prof, z=z_prof, mdef='200m'), '--', 'Hernquist'),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: enclosed mass fraction
plt.sca(axes[0])
plt.loglog()
plt.xlabel(r'$r / R_{\rm vir}$')
plt.ylabel(r'$M(<r) / M_{\rm vir}$')
plt.xlim(rR_prof[0], rR_prof[-1])
plt.ylim(1e-5, 3.0)
plt.axhline(1.0, ls=':', color='gray', lw=0.8)
plt.axvline(1.0, ls=':', color='gray', lw=0.8)

for prf, ls, label in profiles:
    M_enc = prf.enclosedMass(r_prof)
    plt.plot(rR_prof, M_enc / M_prof, ls=ls, lw=1.5, label=label)

plt.legend(fontsize=9)
plt.title(r'Enclosed mass ($M_{200m} = 10^{14}\,h^{-1}\,M_\odot$)')

# Right: circular velocity
plt.sca(axes[1])
plt.xscale('log')
plt.xlabel(r'$r / R_{\rm vir}$')
plt.ylabel(r'$V_c\ ({\rm km/s})$')
plt.xlim(rR_prof[0], rR_prof[-1])

for prf, ls, label in profiles:
    Vc = prf.circularVelocity(r_prof)
    plt.plot(rR_prof, Vc, ls=ls, lw=1.5, label=label)

plt.axvline(1.0, ls=':', color='gray', lw=0.8)
plt.legend(fontsize=9)
plt.title('Circular velocity')

plt.tight_layout()
plt.show()

---

### Exercise 2: The NFW Profile by Hand

In this exercise, you will implement the NFW profile from scratch and compute the circular velocity curve. This will build intuition for how the profile parameters map to observable quantities.

**Given:** A halo with mass $M_{200m}$ and concentration $c$.

**Steps:**
1. Compute the virial radius $R_{200m}$ (provided by Colossus) and the scale radius $r_s = R_{200m} / c$.
2. Compute the scale density $\rho_s$ from the enclosed mass condition. For NFW, the enclosed mass is:
$$M(<r) = 4\pi \rho_s r_s^3 \left[\ln\left(\frac{r_s + r}{r_s}\right) - \frac{r}{r_s + r}\right]$$
Setting $M(<R_{200m}) = M_{200m}$ gives you $\rho_s$.
3. Compute $\rho(r)$, $M(<r)$, and $V_c(r) = \sqrt{GM(<r)/r}$ over a range of radii.
4. Compare your result to the Colossus NFW profile.

**Expected results:**
- Your profile should match the Colossus output essentially exactly (to numerical precision).
- The circular velocity curve should peak at $r \approx 2.16\, r_s$ and then decline.
- **Question:** How does the peak circular velocity $V_{\rm max}$ scale with halo mass? Is it linear? (Hint: think about how $R_{\rm vir} \propto M^{1/3}$ and how concentration enters.)

In [ ]:
# Exercise 2: NFW profile from scratch

M_vir = 1e14       # h^-1 Msun
c_vir = 5.0
z_ex = 0.0
mdef = '200m'

# Get R_200m from Colossus (in kpc/h)
R_vir = mass_so.M_to_R(M_vir, z_ex, mdef)
print('R_200m = %.1f kpc/h' % R_vir)

# Step 1: Scale radius
r_s = ...  # FILL IN: R_vir / c_vir
print('r_s = %.1f kpc/h' % r_s)

# Step 2: Scale density
# The total mass enclosed within R_vir is M_vir.
# For NFW: M(<R) = 4*pi*rho_s*r_s^3 * [ln(1+c) - c/(1+c)]
rho_s = ...  # FILL IN: M_vir / (4.0 * np.pi * r_s**3 * (np.log(1.0 + c_vir) - c_vir / (1.0 + c_vir)))
print('rho_s = %.3e h^2 Msun / kpc^3' % rho_s)

# Radial array
r = np.logspace(np.log10(0.5), np.log10(5.0 * R_vir), 200)  # kpc/h
x = r / r_s

# Step 3a: Density profile
rho_nfw = ...  # FILL IN: rho_s / (x * (1.0 + x)**2)

# Step 3b: Enclosed mass
M_enc = ...  # FILL IN: 4.0 * np.pi * rho_s * r_s**3 * (np.log(1.0 + x) - x / (1.0 + x))

# Step 3c: Circular velocity (constants.G is in kpc km^2 / Msun s^2)
V_c = ...  # FILL IN: np.sqrt(constants.G * M_enc / r)

# Step 4: Compare to Colossus
prf_col = profile_nfw.NFWProfile(M=M_vir, c=c_vir, z=z_ex, mdef=mdef)
rho_col = prf_col.density(r)
Vc_col = prf_col.circularVelocity(r)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Density
plt.sca(axes[0])
plt.loglog()
plt.xlabel(r'$r\ ({\rm kpc}/h)$')
plt.ylabel(r'$\rho\ (h^2\, M_\odot\, {\rm kpc}^{-3})$')
plt.plot(r, rho_nfw, '-', lw=2, label='Your NFW')
plt.plot(r, rho_col, '--', lw=2, label='Colossus')
plt.axvline(R_vir, ls=':', color='gray', lw=0.8, label=r'$R_{200m}$')
plt.legend(fontsize=9)
plt.title('Density')

# Enclosed mass
plt.sca(axes[1])
plt.loglog()
plt.xlabel(r'$r\ ({\rm kpc}/h)$')
plt.ylabel(r'$M(<r)\ (h^{-1}\, M_\odot)$')
M_enc_col = prf_col.enclosedMass(r)
plt.plot(r, M_enc, '-', lw=2, label='Your NFW')
plt.plot(r, M_enc_col, '--', lw=2, label='Colossus')
plt.axvline(R_vir, ls=':', color='gray', lw=0.8)
plt.legend(fontsize=9)
plt.title('Enclosed mass')

# Circular velocity
plt.sca(axes[2])
plt.xlabel(r'$r\ ({\rm kpc}/h)$')
plt.ylabel(r'$V_c\ ({\rm km/s})$')
plt.semilogx()
plt.plot(r, V_c, '-', lw=2, label='Your NFW')
plt.plot(r, Vc_col, '--', lw=2, label='Colossus')
plt.axvline(R_vir, ls=':', color='gray', lw=0.8)
plt.legend(fontsize=9)
plt.title('Circular velocity')

plt.tight_layout()
plt.show()

---

## Comparison to N-body Simulations

The Press-Schechter formalism makes several simplifying assumptions (spherical collapse, Gaussian smoothing, a sharp $\delta_c$ threshold). How well does it actually work? The answer comes from **cosmological N-body simulations**, which follow the gravitational evolution of billions of dark matter particles from the linear regime ($z \sim 100$) to the present day.

We compare to N-body simulations from the **Erebos suite** ([Diemer & Kravtsov 2015](https://ui.adsabs.harvard.edu/abs/2015ApJ...799..108D/abstract)), which span a range of box sizes and resolutions. We also compare to the **Tinker et al. (2008)** fitting function, which was calibrated directly against simulations.

*Note: the Erebos simulations use the Bolshoi cosmology, so we briefly switch cosmologies for this comparison.*

In [ ]:
# N-body comparison (demonstration)
# Switch to Bolshoi cosmology used by the Erebos simulations
cosmo_bol = cosmology.setCosmology('bolshoi')

sim_names = ['l2000-bol', 'l1000-bol', 'l0500-bol', 'l0250-bol', 'l0125-bol', 'l0063-bol']
z_nb = 0.0
cmap_nb = plt.get_cmap('viridis_r')

fig, axs = plt.subplots(2, 1, height_ratios=[1.0, 0.5], figsize=(6, 6))
plt.subplots_adjust(hspace=0.05)

plt.sca(axs[0])
plt.loglog()
plt.ylim(1e-8, 1e0)
axs[0].set_xticklabels([])
plt.ylabel(r'$dn / d\ln(M)\ (h^3\, {\rm Mpc}^{-3})$')

plt.sca(axs[1])
plt.xscale('log')
plt.ylim(0.3, 1.7)
plt.xlabel(r'$M_{\rm vir}\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$n / n_{\rm Tinker08}$')

for ax in axs:
    plt.sca(ax)
    plt.xlim(4e9, 4e15)
    ax.xaxis.set_major_locator(LogLocator(numticks=100))
    ax.xaxis.set_minor_locator(LogLocator(numticks=100, subs=np.arange(2, 10)))

# Analytic mass functions
comp_M = 10**np.linspace(9.0, 15.5, 50)
comp_mf = mass_function.massFunction(comp_M, z_nb, q_in='M', q_out='dndlnM',
                                      model='tinker08', mdef='vir')
ps_mf = mass_function.massFunction(comp_M, z_nb, q_in='M', q_out='dndlnM', model='press74')

# Load simulation data
try:
    for i, sn in enumerate(sim_names):
        c = cmap_nb(float(i) / (len(sim_names) - 1.0))
        mf, bin_centers = routines.getMassFunction(sn, z_nb)

        plt.sca(axs[0])
        plt.plot(bin_centers, mf, c=c, lw=1.2, label=r'${\rm %s}$' % sn[:-4])

        plt.sca(axs[1])
        comp_interp = 10**np.interp(np.log10(bin_centers), np.log10(comp_M), np.log10(comp_mf))
        plt.plot(bin_centers, mf / comp_interp, c=c)

except Exception as e:
    print('Could not load simulation data: %s' % str(e))
    print('Showing analytic models only.')

# Add analytic models
plt.sca(axs[0])
plt.plot(comp_M, comp_mf, '--', color='gray', label='Tinker 08')
plt.plot(comp_M, ps_mf, '-.', color='firebrick', lw=1.0, label='Press-Schechter')
plt.legend(fontsize=8, ncol=2)

plt.sca(axs[1])
plt.axhline(1.0, ls='--', color='gray')
plt.plot(comp_M, ps_mf / comp_mf, '-.', color='firebrick')

plt.sca(axs[0])
plt.title('Mass function: simulations vs. analytic models')
plt.show()

# Switch back to Planck18
cosmo = cosmology.setCosmology('planck18')

### Mass Function Fitting Functions

The Press-Schechter formula captures the qualitative features of the mass function but is known to **overpredict** at low masses and **underpredict** at the exponential high-mass tail by $\sim 50\%$. Understanding *why* it fails is physically instructive:

- **Low-mass overprediction:** The "cloud-in-cloud" problem -- PS counts small regions above threshold even when they are embedded in a larger collapsing region. They should be counted as part of the larger halo, not as independent objects.
- **High-mass underprediction:** Spherical collapse is too restrictive. Real overdensities are triaxial and collapse first along their shortest axis (ellipsoidal collapse), which requires a *lower* effective barrier. This makes it easier to form massive halos.

Several improved fitting functions have been calibrated against N-body simulations:

- **Sheth & Tormen (1999):** Incorporates ellipsoidal (rather than spherical) collapse, using a modified barrier $\delta_c \to \sqrt{a}\,\delta_c$ with $a \approx 0.707$. This naturally fixes both the low-mass and high-mass discrepancies.
- **Tinker et al. (2008):** Calibrated against a large suite of simulations across multiple mass definitions and redshifts. Currently the most widely used model for precision cosmology.

We compare these three models at $z = 0$ to see how they differ.

In [ ]:
# Compare mass function fitting functions at z = 0
models = [('press74', 'Press-Schechter', '-', 'C0'),
          ('sheth99', 'Sheth-Tormen 99', '--', 'C1'),
          ('tinker08', 'Tinker+ 2008', '-.', 'C2')]

M_comp = 10**np.linspace(8.0, 16.0, 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: absolute mass functions
plt.sca(axes[0])
plt.loglog()
plt.xlim(M_comp[0], M_comp[-1])
plt.ylim(1e-8, 1e1)
plt.xlabel(r'$M\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$dn / d\ln(M)\ (h^3\, {\rm Mpc}^{-3})$')
plt.grid(lw=0.5, ls=':')

mf_dict = {}
for model, label, ls, color in models:
    mdef_kw = {'mdef': '200m'} if model == 'tinker08' else {}
    mf = mass_function.massFunction(M_comp, 0.0, q_in='M', q_out='dndlnM', model=model, **mdef_kw)
    mf_dict[model] = mf
    plt.plot(M_comp, mf, ls=ls, color=color, lw=1.8, label=label)

plt.legend(fontsize=9)
plt.title('Mass function models at z = 0')

# Right: ratio to Tinker+2008
plt.sca(axes[1])
plt.xscale('log')
plt.xlim(M_comp[0], M_comp[-1])
plt.ylim(0.3, 2.5)
plt.xlabel(r'$M\ (h^{-1}\, M_\odot)$')
plt.ylabel(r'$dn/d\ln M\ /\ dn/d\ln M_{\rm Tinker08}$')
plt.grid(lw=0.5, ls=':')
plt.axhline(1.0, ls='--', color='gray', lw=0.8)

for model, label, ls, color in models:
    ratio = mf_dict[model] / mf_dict['tinker08']
    plt.plot(M_comp, ratio, ls=ls, color=color, lw=1.8, label=label)

plt.legend(fontsize=9)
plt.title('Ratio to Tinker+ 2008')

plt.tight_layout()
plt.show()

---

## The Milky Way Rotation Curve

In Lecture 14, we saw that the stellar mass of the Milky Way alone cannot explain the observed flat rotation curve -- the circular velocity stays roughly constant at $V_c \approx 220$ km/s from the solar circle out to at least 20 kpc, whereas the visible matter alone would predict a Keplerian decline $V_c \propto r^{-1/2}$.

This is one of the strongest pieces of evidence for **dark matter** on galactic scales. The visible disk and bulge dominate the gravitational potential in the inner few kpc, but beyond the solar circle, a massive dark matter halo must take over. We now have the tools -- specifically the NFW profile and the concentration-mass relation -- to quantitatively model this halo contribution.

The total circular velocity is the quadrature sum of contributions from each mass component:

$$V_{\rm tot}(R) = \sqrt{V_{\rm disk}^2 + V_{\rm bulge}^2 + V_{\rm halo}^2}$$

where:
- **Disk**: exponential surface density profile $\Sigma(R) = \Sigma_0 \exp(-R/R_d)$, with $V_{\rm disk}$ given by Binney & Tremaine (Eq. 2.165) involving modified Bessel functions. The disk dominates at $R \sim 2$-$3\, R_d$.
- **Bulge**: Hernquist profile (from Colossus). Dominates in the inner $\sim 1$ kpc.
- **Halo**: NFW profile, using the concentration-mass relation to set $c$ for a given $M_{\rm halo}$. Dominates beyond $\sim 15$ kpc and extends to $\sim 200$ kpc.

The fact that these three components -- with independently constrained parameters -- sum to a flat rotation curve is a non-trivial success of the CDM framework (though the detailed "conspiracy" between disk and halo contributions remains an active area of research).

### Exercise 3: Milky Way Rotation Curve Decomposition

In this exercise, you will decompose the observed MW rotation curve into contributions from the disk, bulge, and dark matter halo. This is one of the classic analyses in galactic dynamics.

**Given parameters** (Licquia & Newman 2016; Ninkovic 2017):
- Disk mass: $M_{\rm disk} = 4.8 \times 10^{10}\,M_\odot$, scale length $R_d = 2.64$ kpc
- Bulge mass: $M_{\rm bulge} = 1.85 \times 10^{10}\,M_\odot$, scale radius $a = 0.3$ kpc
- Halo mass: $M_{\rm halo} = 10^{12}\,M_\odot$

**Steps:**
1. Compute $V_{\rm disk}$ using the Binney & Tremaine formula (provided). This involves modified Bessel functions $I_n$ and $K_n$ evaluated at $y = R/(2R_d)$.
2. Compute $V_{\rm bulge}$ using a Hernquist profile (provided). Note: $V_{\rm bulge}(r) = \sqrt{GM_{\rm bulge}\, r / (r + a)^2}$.
3. **FILL IN:** Compute $V_{\rm halo}$ from an NFW profile. Use the concentration-mass relation (`concentration.concentration()`) to get $c$ for the given halo mass, then compute the NFW enclosed mass and circular velocity.
4. **FILL IN:** Compute $V_{\rm tot} = \sqrt{V_{\rm disk}^2 + V_{\rm bulge}^2 + V_{\rm halo}^2}$ and compare to the observed ~220 km/s.

**Expected results and interpretation:**
- The disk contribution should peak at $R \approx 2.2\, R_d \approx 6$ kpc and then decline.
- The bulge contribution is significant only in the inner few kpc.
- The halo contribution should rise and flatten, dominating beyond $\sim 15$ kpc.
- The total $V_{\rm tot}$ should be approximately flat at $\sim 220$ km/s from 5-20 kpc.
- **Question:** What fraction of the mass within the solar circle ($R \approx 8$ kpc) is dark matter? What about within 50 kpc?

In [ ]:
# Exercise 3: MW Rotation Curve Decomposition

# Parameters
M_disk = 4.8e10       # Msun
R_d = routines.Rd_MW  # kpc
M_bulge = 1.85e10     # Msun
R_bulge = 0.3         # kpc
M_halo = 1.0e12       # Msun

# Load rotation curve data (Sofue 2020)
d = np.loadtxt('data/rotation_curve_sofue_2020.txt', unpack=True)
# d[0] = R (kpc), d[1] = V (km/s), d[2] = V_err (km/s)

# Radial array
R = np.linspace(1e-5, 100.0, 200)  # kpc

# ── Disk: exponential disk (BT Eq. 2.165) ──
# (provided — uses modified Bessel functions)
Sigma_0 = M_disk / (2.0 * np.pi * R_d**2)
y = R / (2.0 * R_d)
I_0 = scipy.special.iv(0, y)
I_1 = scipy.special.iv(1, y)
K_0 = scipy.special.kn(0, y)
K_1 = scipy.special.kn(1, y)
V_disk = np.sqrt(4.0 * np.pi * constants.G * Sigma_0 * R_d * y**2 * (I_0 * K_0 - I_1 * K_1))

# ── Bulge: Hernquist profile (provided) ──
h = cosmo.h
prf_b = profile_hernquist.HernquistProfile(rhos=1.0, rs=R_bulge * h)
M_norm = prf_b.enclosedMass(1000.0) / h
prf_b = profile_hernquist.HernquistProfile(rhos=M_bulge / M_norm, rs=R_bulge * h)
V_bulge = prf_b.circularVelocity(R * h)

# ── Halo: NFW profile (FILL IN) ──
# Get concentration for this halo mass
c_halo = ...  # FILL IN: concentration.concentration(M_halo * h, 'vir', 0.0)

# Virial radius (convert from kpc/h to kpc)
R_halo = mass_so.M_to_R(M_halo * h, 0.0, 'vir') / h  # kpc
r_s_halo = R_halo / c_halo

# Scale density
rho_s_halo = ...  # FILL IN: M_halo / (4.0 * np.pi * r_s_halo**3 * (np.log(1.0 + c_halo) - c_halo / (1.0 + c_halo)))

# Enclosed mass and circular velocity
x_halo = R / r_s_halo
M_enc_halo = 4.0 * np.pi * r_s_halo**3 * rho_s_halo * (np.log(1.0 + x_halo) - x_halo / (1.0 + x_halo))
V_halo = ...  # FILL IN: np.sqrt(constants.G * M_enc_halo / R)

# ── Total circular velocity ──
V_tot = ...  # FILL IN: np.sqrt(V_disk**2 + V_bulge**2 + V_halo**2)

# ── Plot ──
plt.figure(figsize=(7, 5))
plt.xlabel(r'$R\ ({\rm kpc})$')
plt.ylabel(r'$V_c\ ({\rm km/s})$')

plt.axvline(8.0, ls='--', color='gray', lw=0.8, label='Solar position')
plt.errorbar(d[0], d[1], d[2], fmt='o', ms=2.0, color='gray', lw=0.5, label='Observations')

plt.plot(R, V_disk, '-.', label='Exponential disk', lw=1.2)
plt.plot(R, V_bulge, '--', label='Bulge', lw=1.2)
plt.plot(R, V_halo, ':', label='NFW halo', lw=1.2)
plt.plot(R, V_tot, '-', label='Total', lw=2, zorder=100)

plt.xlim(-2, 100)
plt.ylim(0, 300)
plt.legend(labelspacing=0.15, borderpad=0.2)
plt.title('Milky Way rotation curve decomposition')
plt.tight_layout()
plt.show()

print('Halo concentration: c = %.2f' % c_halo)
print('Virial radius: R_vir = %.1f kpc' % R_halo)
print('Scale radius: r_s = %.1f kpc' % r_s_halo)

---

## Summary

1. **Spherical collapse** predicts that overdensities collapse when the linearly extrapolated overdensity reaches $\delta_c = 1.686$. This threshold, combined with the variance $\sigma(M)$ of the linear density field, determines which mass scales are collapsing at each epoch. Structure forms bottom-up: low-mass halos assemble first.

2. The **Press-Schechter mass function** provides an analytic prediction for the abundance of halos by combining Gaussian statistics with the spherical collapse threshold. It captures the qualitative features but overpredicts low-mass halos (cloud-in-cloud problem) and underpredicts high-mass halos (spherical collapse is too restrictive) by $\sim 50\%$. More accurate fitting functions (Sheth-Tormen 1999; Tinker+2008) are calibrated against N-body simulations.

3. Halo density profiles are well described by the **NFW profile** $\rho \propto r^{-1}(1+r/r_s)^{-2}$, with an inner cusp ($r^{-1}$) from early violent relaxation and an outer envelope ($r^{-3}$) from recent accretion. The universal **concentration-mass relation** encodes formation history: early-forming (low-mass) halos are more concentrated because they formed when the universe was denser.

4. The **flat rotation curve** of the Milky Way requires a dark matter halo with $M \sim 10^{12}\,M_\odot$. The halo dominates the gravitational potential beyond the solar circle, and the disk-halo "conspiracy" that produces a flat total $V_c(R)$ is a key prediction of CDM.

**Next time:** The galaxy-halo connection -- how galaxies populate dark matter halos (CFN Ch. 5).